# Visualize your data

<img src="./media/data.gif" width="480" height="360">

Visualize your action based on the reconstructed simulation scene. 

The main simulation is replaying the action.

The overlayed images on the top right and bottom right are from the dataset. 

In [14]:
import sys
sys.path.append('/home/student/Desktop/lerobot-papras')
from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata
import numpy as np
from lerobot.datasets.utils import write_json, serialize_dict

root ='/home/student/Desktop/lerobot-papras/nov_19_e20'
root = '/home/student/Desktop/lerobot-papras/dec_8_all'
# dataset = LeRobotDataset('nov_19_e20', root=root) # if youu want to use the example data provided, root = './demo_data_example' instead!
dataset = LeRobotDataset('dec_8_all', root=root)

In [2]:
roots = ['/home/student/Desktop/lerobot-papras/dec_8_e60_v1', '/home/student/Desktop/lerobot-papras/dec_8_e60_v2', '/home/student/Desktop/lerobot-papras/dec_8_e60_v3']

## Load Dataset

In [3]:
# Select an episode index that you want to visualize
import torch
# Accessing an index now returns a stack for the specified key(s)
sample = dataset[0]
print(sample["observation.wrist_image"].shape)  # [T, C, H, W], where T=3

# 4) Wrap with a DataLoader for training
batch_size = 16
data_loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size)


torch.Size([3, 256, 256])


In [4]:
print(len(dataset)//16, len(data_loader))

316 317


In [5]:
type(dataset.meta.episodes)

datasets.arrow_dataset.Dataset

In [15]:
class EpisodeSampler(torch.utils.data.Sampler):
    """
    Sampler for a single episode
    """
    def __init__(self, dataset: LeRobotDataset, episode_index: int):
        episode = dataset.meta.episodes.filter(lambda example: example["episode_index"]==episode_index)[0]
        from_idx = episode["dataset_from_index"],
        to_idx = episode["dataset_to_index"]
        # print(from_idx, to_idx)
        self.frame_ids = range(from_idx[0], to_idx)

    def __iter__(self):
        return iter(self.frame_ids)

    def __len__(self) -> int:
        return len(self.frame_ids)
    
episode_index = 0

episode_sampler = EpisodeSampler(dataset, episode_index)
episode_dataloader = torch.utils.data.DataLoader(
    dataset,
    num_workers=1,
    batch_size=1,
    sampler=episode_sampler,
)


Filter:   0%|          | 0/60 [00:00<?, ? examples/s]

## Visualize your Dataset on Simulation

In [7]:
from mujoco_env.papras7dof_env import PaprasEnv
xml_path = './asset/papras_scene.xml'
# pdb.set_trace()
SEED = 0
# Define the environment
PnPEnv = PaprasEnv(xml_path, seed = SEED, action_type='joint_angle', state_type = 'joint_angle')


-----------------------------------------------------------------------------
name:[Tabletop] dt:[0.002] HZ:[500]
 n_qpos:[39] n_qvel:[35] n_qacc:[35] n_ctrl:[11]
 integrator:[RK4]

n_body:[31]
 [0/31] [world] mass:[0.00]kg
 [1/31] [front_object_table] mass:[1.00]kg
 [2/31] [camera] mass:[0.00]kg
 [3/31] [camera2] mass:[0.00]kg
 [4/31] [camera3] mass:[0.00]kg
 [5/31] [robot1/link1] mass:[0.86]kg
 [6/31] [robot1/link2] mass:[0.95]kg
 [7/31] [robot1/link3] mass:[0.50]kg
 [8/31] [robot1/link4] mass:[0.60]kg
 [9/31] [robot1/link5] mass:[1.16]kg
 [10/31] [robot1/link6] mass:[0.45]kg
 [11/31] [robot1/link7] mass:[0.43]kg
 [12/31] [robot1/end_link] mass:[0.02]kg
 [13/31] [robot1/wrist_link] mass:[0.00]kg
 [14/31] [robot1/gripper_main_link] mass:[0.24]kg
 [15/31] [robot1/gripper_link] mass:[0.07]kg
 [16/31] [robot1/gripper_link_r2] mass:[0.02]kg
 [17/31] [robot1/gripper_link_l1] mass:[0.07]kg
 [18/31] [robot1/gripper_link_l2] mass:[0.02]kg
 [19/31] [robot1/end_effector_link] mass:[0.00]kg
 [2

In [8]:
step = 0
iter_episode = iter(episode_dataloader)
PnPEnv.reset()

while PnPEnv.env.is_viewer_alive():
    PnPEnv.step_env()
    if PnPEnv.env.loop_every(HZ=20):
        # Get the action from dataset
        data = next(iter_episode)
        if step == 0:
            # Reset the object pose based on the dataset
            PnPEnv.set_obj_pose(data['obj_init'][0,:3], data['obj_init'][0,3:])
        # Get the action from dataset
        action = data['action'].numpy()
        obs = PnPEnv.step(action[0])

        # Visualize the image from dataset to rgb_overlay
        PnPEnv.rgb_agent = data['observation.image'][0].numpy()*255
        PnPEnv.rgb_ego = data['observation.wrist_image'][0].numpy()*255
        PnPEnv.rgb_agent = PnPEnv.rgb_agent.astype(np.uint8)
        PnPEnv.rgb_ego = PnPEnv.rgb_ego.astype(np.uint8)
        # 3 256 256 -> 256 256 3
        PnPEnv.rgb_agent = np.transpose(PnPEnv.rgb_agent, (1,2,0))
        PnPEnv.rgb_ego = np.transpose(PnPEnv.rgb_ego, (1,2,0))
        PnPEnv.rgb_side = np.zeros((480, 640, 3), dtype=np.uint8)
        PnPEnv.render()
        step += 1

        if step == len(episode_dataloader):
            # start from the beginning
            iter_episode = iter(episode_dataloader)
            PnPEnv.reset()
            step = 0

DONE INITIALIZATION
DONE INITIALIZATION


KeyboardInterrupt: 

In [8]:
PnPEnv.env.close_viewer()

# User support for classifying episodes for strategy


In [18]:
from mujoco_env.papras7dof_env import PaprasEnv
import glfw
xml_path = './asset/papras_scene.xml'
# pdb.set_trace()
SEED = 0
N = 40
# Define the environment
PnPEnv = PaprasEnv(xml_path, seed = SEED, action_type='joint_angle', state_type = 'joint_angle')
# strategy_classes = np.zeros(N)

def classify_episode(episode_index):

    episode_sampler = EpisodeSampler(dataset, episode_index)
    episode_dataloader = torch.utils.data.DataLoader(
        dataset,
        num_workers=1,
        batch_size=1,
        sampler=episode_sampler,
    )

    step = 0
    iter_episode = iter(episode_dataloader)
    PnPEnv.reset()

    while PnPEnv.env.is_viewer_alive():
        PnPEnv.step_env()
        if PnPEnv.env.loop_every(HZ=20):
            # Get the action from dataset
            data = next(iter_episode)
            if step == 0:
                # Reset the object pose based on the dataset
                PnPEnv.set_obj_pose(data['obj_init'][0,:3], data['obj_init'][0,3:])
            # Get the action from dataset
            action = data['action'].numpy()
            obs = PnPEnv.step(action[0])

            # Visualize the image from dataset to rgb_overlay
            PnPEnv.rgb_agent = data['observation.image'][0].numpy()*255
            PnPEnv.rgb_ego = data['observation.wrist_image'][0].numpy()*255
            PnPEnv.rgb_agent = PnPEnv.rgb_agent.astype(np.uint8)
            PnPEnv.rgb_ego = PnPEnv.rgb_ego.astype(np.uint8)
            # 3 256 256 -> 256 256 3
            PnPEnv.rgb_agent = np.transpose(PnPEnv.rgb_agent, (1,2,0))
            PnPEnv.rgb_ego = np.transpose(PnPEnv.rgb_ego, (1,2,0))
            PnPEnv.rgb_side = np.zeros((480, 640, 3), dtype=np.uint8)
            PnPEnv.render()
            step += 1

            if step == len(episode_dataloader):
                # start from the beginning
                iter_episode = iter(episode_dataloader)
                PnPEnv.reset()
                step = 0
            
            if PnPEnv.env.is_key_pressed_once(key=glfw.KEY_A):
                return 0
            if PnPEnv.env.is_key_pressed_once(key=glfw.KEY_S):
                return 1
            if PnPEnv.env.is_key_pressed_once(key=glfw.KEY_D):
                return 2

for i in range(20, 20+N):
    #0: grip by handle, 1: grip inside, 3:hybrid
    strategy_classes = np.append(strategy_classes, classify_episode(i))
    print(f"Episode {i} classified as {strategy_classes[i]}")

PnPEnv.env.close_viewer()


-----------------------------------------------------------------------------
name:[Tabletop] dt:[0.002] HZ:[500]
 n_qpos:[39] n_qvel:[35] n_qacc:[35] n_ctrl:[11]
 integrator:[RK4]

n_body:[31]
 [0/31] [world] mass:[0.00]kg
 [1/31] [front_object_table] mass:[1.00]kg
 [2/31] [camera] mass:[0.00]kg
 [3/31] [camera2] mass:[0.00]kg
 [4/31] [camera3] mass:[0.00]kg
 [5/31] [robot1/link1] mass:[0.86]kg
 [6/31] [robot1/link2] mass:[0.95]kg
 [7/31] [robot1/link3] mass:[0.50]kg
 [8/31] [robot1/link4] mass:[0.60]kg
 [9/31] [robot1/link5] mass:[1.16]kg
 [10/31] [robot1/link6] mass:[0.45]kg
 [11/31] [robot1/link7] mass:[0.43]kg
 [12/31] [robot1/end_link] mass:[0.02]kg
 [13/31] [robot1/wrist_link] mass:[0.00]kg
 [14/31] [robot1/gripper_main_link] mass:[0.24]kg
 [15/31] [robot1/gripper_link] mass:[0.07]kg
 [16/31] [robot1/gripper_link_r2] mass:[0.02]kg
 [17/31] [robot1/gripper_link_l1] mass:[0.07]kg
 [18/31] [robot1/gripper_link_l2] mass:[0.02]kg
 [19/31] [robot1/end_effector_link] mass:[0.00]kg
 [2

Filter:   0%|          | 0/60 [00:00<?, ? examples/s]

DONE INITIALIZATION
Episode 21 classified as 1.0


Filter:   0%|          | 0/60 [00:00<?, ? examples/s]

DONE INITIALIZATION
Episode 22 classified as 0.0


...

DONE INITIALIZATION
DONE INITIALIZATION
Episode 57 classified as 1.0


Filter:   0%|          | 0/60 [00:00<?, ? examples/s]

DONE INITIALIZATION
Episode 58 classified as 1.0


Filter:   0%|          | 0/60 [00:00<?, ? examples/s]

DONE INITIALIZATION
Episode 59 classified as 2.0


In [19]:
print(strategy_classes)
np.savetxt(root+"/strategy.txt",strategy_classes)

[0. 2. 1. 1. 1. 0. 1. 1. 1. 1. 2. 1. 1. 1. 0. 1. 1. 2. 2. 0. 1. 1. 0. 2.
 1. 1. 0. 1. 2. 2. 2. 1. 1. 1. 2. 2. 1. 1. 1. 1. 2. 2. 0. 1. 2. 2. 2. 1.
 2. 1. 1. 1. 1. 2. 1. 2. 1. 1. 1. 2.]


```
lerobot-edit-dataset \
    --repo_id lerobot/pusht \
    --new_repo_id lerobot/pusht_after_deletion \
    --operation.type delete_episodes \
    --operation.episode_indices "[0, 2, 5]"
```

In [20]:
len(strategy_classes)

60

In [ ]:
test_strategies = np.loadtxt('/home/student/Desktop/lerobot-papras/nov_19_e20/strategy.txt')
print(len(test_strategies))
grip_inside = np.argwhere(test_strategies == 1)[:,0]
print(list(grip_inside))

grip_handle = np.argwhere(test_strategies == 0)[:,0]
print(list(grip_handle))
"""
lerobot-edit-dataset \
    --repo_id lerobot/nov_19_e20\
    --root /home/student/Desktop/lerobot-papras/nov_19_e20\
    --operation.type split\
    --operation.splits '{"grip_outside": [0, 4, 5, 6, 8, 9, 10, 11, 12, 15, 17], "grip_handle": [1, 2, 3, 13, 16]}'

"""


20
[0, 4, 5, 6, 8, 9, 10, 11, 12, 15, 17]
[1, 2, 3, 13, 16]


'\nlerobot-edit-dataset     --repo_id lerobot/nov_19_e20    --root /home/student/Desktop/lerobot-papras/nov_19_e20    --new_repo_id lerobot/grip_inside     --operation.type delete_episodes     --operation.episode_indices "[0, 4, 5, 6, 8, 9, 10, 11, 12, 15, 17]"\n\n'

In [31]:
test_strategies = strategy_classes
print(len(test_strategies))
grip_inside = np.argwhere(test_strategies == 1)[:,0]
print(list(grip_inside))

grip_handle = np.argwhere(test_strategies == 0)[:,0]
print(list(grip_handle))
"""
lerobot-edit-dataset \
    --repo_id lerobot/dec_8_all\
    --root /home/student/Desktop/lerobot-papras/dec_8_all\
    --operation.type split\
    --operation.splits '{"grip_outside": [2, 3, 4, 6, 7, 8, 9, 11, 12, 13, 15, 16, 20, 21, 24, 25, 27, 31, 32, 33, 36, 37, 38, 39, 43, 47, 49, 50, 51, 52, 54, 56, 57, 58], "grip_handle": [0, 5, 14, 19, 22, 26, 42]}'

"""


60
[2, 3, 4, 6, 7, 8, 9, 11, 12, 13, 15, 16, 20, 21, 24, 25, 27, 31, 32, 33, 36, 37, 38, 39, 43, 47, 49, 50, 51, 52, 54, 56, 57, 58]
[0, 5, 14, 19, 22, 26, 42]


'\nlerobot-edit-dataset     --repo_id lerobot/dec_8_all    --root /home/student/Desktop/lerobot-papras/dec_8_all    --operation.type split    --operation.splits \'{"grip_outside": [2, 3, 4, 6, 7, 8, 9, 11, 12, 13, 15, 16, 20, 21, 24, 25, 27, 31, 32, 33, 36, 37, 38, 39, 43, 47, 49, 50, 51, 52, 54, 56, 57, 58], "grip_handle": [0, 5, 14, 19, 22, 26, 42]}\'\n\n'

### [Optional] Save Stats.json for other versions

In [7]:
stats = dataset.meta.stats
PATH = dataset.root / 'meta' / 'stats.json'
stats = serialize_dict(stats)

write_json(stats, PATH)